# Model Selection Analysis

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import sklearn
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import f1_score, roc_auc_score

import sys
sys.path.append('..')
from data_pipeline import ETL_Pipeline
from dataset import Fraud_Dataset
from metrics import Metrics


### Extract, Transform, and Load the Data

In [2]:
etlpipe = ETL_Pipeline()
fpth = Path('../data/transactions-1.csv')
etlpipe.extract(fpth)
etlpipe.transform()
pdpth = Path('../data/procdata.csv')
etlpipe.load(pdpth)

### Split the Data into Training, Validation, and Testing Sets

In [3]:
# 10% for testing.  The remaining is split into 5 folds for CV.
datsets = Fraud_Dataset(pdpth, (0.72, 0.18, 0.1))
X_test, Y_test = datsets.get_testing_dataset()
X_trains, X_vals, Y_trains, Y_vals = datsets.get_nfoldCV_datasets()

### Validate the XGBoost, Random Forest, and Quadratic Discriminant Analysis Classifiers.

In [4]:
xgbc = XGBClassifier(tree_method='gpu_hist', max_depth=8, objective='binary:logistic')
rfc = RandomForestClassifier(n_jobs=-1, min_samples_leaf=2)
qdac = QuadraticDiscriminantAnalysis()


In [5]:
def compare_models(xgbc, qdac, rfc):
    for clf, name in zip([xgbc, qdac, rfc], ['XGB', 'QDA', 'RF']):
        F1_trains = []
        F1_vals = []
        ROCAUC_trains = []
        ROCAUC_vals = []
        print('{} classifier performance:'.format(name))
        for i, (X_train, X_val, Y_train, Y_val) in\
                enumerate(zip(X_trains, X_vals, Y_trains, Y_vals)):
            clf.fit(X_train, Y_train)
            Y_pred_train = clf.predict(X_train)
            Y_pred_val = clf.predict(X_val)

            # we will use F1 score as the first performance metric
            F1_train = f1_score(Y_train, Y_pred_train)
            F1_val = f1_score(Y_val, Y_pred_val)

            # ROC AUC score will be the second performance metric
            ROCAUC_train = roc_auc_score(Y_train, Y_pred_train)
            ROCAUC_val = roc_auc_score(Y_val, Y_pred_val)

            print('fold {} complete'.format(i))

            F1_trains.append(F1_train)
            F1_vals.append(F1_val)
            ROCAUC_trains.append(ROCAUC_train)
            ROCAUC_vals.append(ROCAUC_val)

        print('mean F1_score train: {}, val: {}\nmean ROCAUC train: {}, val: {}\n'.\
              format(np.mean(F1_train), np.mean(F1_val),\
                     np.mean(ROCAUC_train), np.mean(ROCAUC_val)))

compare_models(xgbc, qdac, rfc)

XGB classifier performance:
fold 0 complete
fold 1 complete
fold 2 complete
fold 3 complete
fold 4 complete
mean F1_score train: 0.9659722737044999, val: 0.8970588235294118
mean ROCAUC train: 0.968763063908454, val: 0.9213212675534856

QDA classifier performance:


/usr/local/lib/python3.10/dist-packages/sklearn/discriminant_analysis.py:926: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


fold 0 complete


/usr/local/lib/python3.10/dist-packages/sklearn/discriminant_analysis.py:926: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


fold 1 complete


...

fold 2 complete


/usr/local/lib/python3.10/dist-packages/sklearn/discriminant_analysis.py:926: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


fold 3 complete


/usr/local/lib/python3.10/dist-packages/sklearn/discriminant_analysis.py:926: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")


fold 4 complete
mean F1_score train: 0.167940943843923, val: 0.17542016806722688
mean ROCAUC train: 0.5458339329399914, val: 0.5480713874496258

RF classifier performance:
fold 0 complete
fold 1 complete
fold 2 complete
fold 3 complete
fold 4 complete
mean F1_score train: 0.9235993208828523, val: 0.8449511400651466
mean ROCAUC train: 0.9305561276586339, val: 0.8732905803899678



**The XGBoost Classifier had the best mean performance across n-fold CV.**

In [6]:
X_train, Y_train = datsets.get_noval_training_dataset()
xgbc.fit(X_train, Y_train)
Y_pred_test = xgbc.predict(X_test)

metrics = Metrics(Y_test, Y_pred_test)
rpth = Path('../results/report.txt')
metrics.generate_report(rpth)

Model Testing Results:

note:
sensitivity = recall of label 1
specificity = recall of label 0
_____________________________________________________
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    184275
           1       0.96      0.83      0.89       965

    accuracy                           1.00    185240
   macro avg       0.98      0.91      0.94    185240
weighted avg       1.00      1.00      1.00    185240

ROC-AUC score: 0.9144128052590917
